In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Load the dataset
try:
    df = pd.read_csv("/content/developer_productivity.csv", encoding='latin-1')
except FileNotFoundError:
    print("Error: spam.csv not found. Please upload the file or provide the correct path.")
    # Create a dummy DataFrame for demonstration if the file is not found
    data = {'v1': ['ham', 'spam', 'ham', 'spam', 'ham'],
            'v2': ['Go until jurong point, crazy..', 'Free entry in 2 a wkly comp to win FA Cup finals tkts 21st May 2005.', 'U dun say so early hor... U c already then say...', 'Had your mobile 11 months or more? U R entitled to Update to the latest colour mobiles with camera for Free!', 'Nah I dont think he goes to usf, he only walks around here.']}
    df = pd.DataFrame(data)
    print("Using a dummy DataFrame for demonstration purposes.")

# Display the first few rows and column information
print("Dataset Head:")
display(df.head())
print("\nDataset Info:")
df.info()

Dataset Head:


,developer_id,commits_per_week,lines_added,lines_deleted,files_changed,bugs_reported,code_review_comments,avg_review_time_hours,test_coverage_percent,deployment_frequency,late_night_commits,commit_risk
0,dev_07,8,89,54,1,3,0,4.5,67.0,2,1,Medium
1,dev_20,4,80,100,4,1,4,21.0,78.3,1,0,Medium
2,dev_08,9,18,9,9,0,4,4.9,79.9,3,0,Low
3,dev_15,3,33,107,4,0,1,15.9,82.5,2,3,Medium
4,dev_07,3,36,15,5,0,2,1.8,87.3,1,1,Low



Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   developer_id           2000 non-null   object 
 1   commits_per_week       2000 non-null   int64  
 2   lines_added            2000 non-null   int64  
 3   lines_deleted          2000 non-null   int64  
 4   files_changed          2000 non-null   int64  
 5   bugs_reported          2000 non-null   int64  
 6   code_review_comments   2000 non-null   int64  
 7   avg_review_time_hours  2000 non-null   float64
 8   test_coverage_percent  2000 non-null   float64
 9   deployment_frequency   2000 non-null   int64  
 10  late_night_commits     2000 non-null   int64  
 11  commit_risk            2000 non-null   object 
dtypes: float64(2), int64(8), object(2)
memory usage: 187.6+ KB


## Developer Productivity & Code Quality Dashboard

This section will demonstrate how to build a logistic regression model to classify spam messages using the `developer_productivity.csv` dataset.

### Data Preprocessing

1.  **Rename columns**: Rename `v1` to `label` and `v2` to `text` for better readability.
2.  **Handle missing values**: Check and remove any rows with missing data.
3.  **Encode labels**: Convert categorical labels ('ham', 'spam') into numerical format (0, 1).

In [7]:
# The original code was designed for a different dataset (e.g., spam.csv)
# which had 'v1' and 'v2' columns for labels and text, respectively.
# The current dataset 'developer_productivity.csv' has different columns.
# Specifically, 'commit_risk' is the likely target variable.

# Ensure 'df' has the expected 'commit_risk' column.
# If not, attempt to reload the dataset to correct the state.
if 'commit_risk' not in df.columns:
    print("Detected missing 'commit_risk' column. Attempting to reload 'developer_productivity.csv'.")
    try:
        # Assuming the path is still /content/developer_productivity.csv as used in the previous cell.
        df = pd.read_csv("/content/developer_productivity.csv", encoding='latin-1')
        print("Dataset reloaded successfully.")
    except FileNotFoundError:
        print("Error: 'developer_productivity.csv' not found during reload attempt. Cannot proceed with preprocessing.")
        # If reload fails, we cannot proceed meaningfully, so we might want to exit or return.
        # For this context, we will simply stop further processing.
        # This will prevent subsequent KeyErrors.
        # We can add a return here if this were a function, but in a cell, we just let it run.
    except Exception as e:
        print(f"An unexpected error occurred during dataset reload: {e}")

# Proceed with preprocessing only if 'commit_risk' (or 'label') is now present.
if 'commit_risk' in df.columns or 'label' in df.columns:
    # 1. Rename 'commit_risk' to 'label' for consistency with machine learning tasks
    #    If the 'commit_risk' column already exists (which it should now after potential reload)
    if 'commit_risk' in df.columns:
        df = df.rename(columns={'commit_risk': 'label'})
        print("Renamed 'commit_risk' to 'label'.")
    # No else here, as if commit_risk was missing, we tried to reload, and if it's still missing,
    # the subsequent encoding will fail gracefully.

    # 2. Check for missing values
    print("Missing values before dropping:")
    display(df.isnull().sum())

    # Drop rows with any missing values (if any)
    df.dropna(inplace=True)
    print("\nMissing values after dropping:")
    display(df.isnull().sum())

    # 3. Encode the 'label' column
    if 'label' in df.columns and df['label'].dtype == 'object': # Check if 'label' exists and is object type
        risk_mapping = {'Low': 0, 'Medium': 1, 'High': 2}
        # Check if all values in 'label' are present in risk_mapping keys to avoid NaN
        if set(df['label'].unique()).issubset(set(risk_mapping.keys())):
            df['label'] = df['label'].map(risk_mapping)
            print("\n'label' column (Commit Risk) encoded to numerical values.")
        else:
            print("Warning: 'label' column contains unexpected values not in risk_mapping. Encoding skipped.")
    elif 'label' not in df.columns:
        print("Error: 'label' column still not found after processing. Cannot encode.")
    else:
        print("Info: 'label' column is already numerical or not a string type, skipping encoding.")

    print("\nDataFrame after preprocessing:")
    display(df.head())
    if 'label' in df.columns:
        display(df['label'].value_counts())
    else:
        print("Cannot display value counts for 'label' as it was not found.")
else:
    print("Cannot proceed with preprocessing: 'commit_risk' (or 'label') column not found even after reload attempt.")

Detected missing 'commit_risk' column. Attempting to reload 'developer_productivity.csv'.
Dataset reloaded successfully.
Renamed 'commit_risk' to 'label'.
Missing values before dropping:


,0
developer_id,0
commits_per_week,0
lines_added,0
lines_deleted,0
files_changed,0
bugs_reported,0
code_review_comments,0
avg_review_time_hours,0
test_coverage_percent,0
deployment_frequency,0



Missing values after dropping:


,0
developer_id,0
commits_per_week,0
lines_added,0
lines_deleted,0
files_changed,0
bugs_reported,0
code_review_comments,0
avg_review_time_hours,0
test_coverage_percent,0
deployment_frequency,0



'label' column (Commit Risk) encoded to numerical values.

DataFrame after preprocessing:


,developer_id,commits_per_week,lines_added,lines_deleted,files_changed,bugs_reported,code_review_comments,avg_review_time_hours,test_coverage_percent,deployment_frequency,late_night_commits,label
0,dev_07,8,89,54,1,3,0,4.5,67.0,2,1,1
1,dev_20,4,80,100,4,1,4,21.0,78.3,1,0,1
2,dev_08,9,18,9,9,0,4,4.9,79.9,3,0,0
3,dev_15,3,33,107,4,0,1,15.9,82.5,2,3,1
4,dev_07,3,36,15,5,0,2,1.8,87.3,1,1,0


,count
label,
0,1101
1,732
2,167


### Split Data and Feature Extraction

1.  **Split the data**: Divide the dataset into training and testing sets.
2.  **TF-IDF Vectorization**: Convert the text data into numerical feature vectors using TF-IDF (Term Frequency-Inverse Document Frequency).

In [9]:
# Split data into features (X) and target (y)
# The 'developer_productivity.csv' dataset does not have a 'text' column.
# Instead, we will use the numerical features and potentially other categorical features (if any) as X.
# The 'label' column is the target variable (commit_risk).

# Drop non-feature columns like 'developer_id' and the target 'label' to create features (X)
# We'll also drop any remaining object columns that are not suitable as features for now
X = df.drop(columns=['developer_id', 'label'], errors='ignore')

# If there are still object columns in X that need encoding, they should be handled here.
# For now, let's assume all remaining columns are numeric or suitable after previous preprocessing.
# If there are other categorical columns, you might need to use pd.get_dummies or another encoder.

y = df['label']

print(f"Features (X) columns: {X.columns.tolist()}")

# Split the dataset into training and testing sets
# Adjusted test_size to 0.4. Ensure stratify works with multi-class if necessary.
# Check unique values in y to ensure stratification is possible.
if y.nunique() > 1:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)
else:
    print("Warning: Only one class present in y. Stratification skipped.")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)

print(f"Training set size: {len(X_train)}")
print(f"Testing set size: {len(X_test)}")

# For this numerical dataset, TF-IDF Vectorizer is not applicable.
# If text-like features were present, a different column would be used for TfidfVectorizer.
# Since we are using numerical features directly, we don't need TF-IDF.
# The X_train and X_test are already in a suitable format (pandas DataFrames) for many models.

# If the user intended to use text data from another source, that would need to be loaded and processed separately.

print(f"\nShape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")

Features (X) columns: ['commits_per_week', 'lines_added', 'lines_deleted', 'files_changed', 'bugs_reported', 'code_review_comments', 'avg_review_time_hours', 'test_coverage_percent', 'deployment_frequency', 'late_night_commits']
Training set size: 1200
Testing set size: 800

Shape of X_train: (1200, 10)
Shape of X_test: (800, 10)


### Train Logistic Regression Model

Initialize and train a Logistic Regression model on the TF-IDF transformed training data.

In [42]:
# Initialize the Logistic Regression model
model = LogisticRegression(solver='liblinear', random_state=42)

# Train the model
# The previous error occurred because 'X_train_tfidf' was not defined as TF-IDF is not applicable to this numerical dataset.
# We should use 'X_train' directly for training.
model.fit(X_train, y_train)

print("Logistic Regression model trained successfully!")

Logistic Regression model trained successfully!


### Model Evaluation

Evaluate the trained model's performance on the test set using accuracy and a classification report.

In [14]:
# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Model Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(report)


Model Accuracy: 0.7725

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.92      0.86       440
           1       0.70      0.67      0.68       293
           2       0.86      0.28      0.43        67

    accuracy                           0.77       800
   macro avg       0.79      0.62      0.66       800
weighted avg       0.77      0.77      0.76       800



### Additional Model Training and Evaluation

This section will train and evaluate additional machine learning models for spam detection: Decision Tree, Support Vector Machine (SVM), and K-Nearest Neighbors (KNN).

#### 1. Decision Tree Classifier

In [15]:
from sklearn.tree import DecisionTreeClassifier

# Initialize the Decision Tree Classifier
dt_model = DecisionTreeClassifier(random_state=42)

# Train the model
# The error occurred because 'X_train_tfidf' is not defined. We should use 'X_train' for this numerical dataset.
dt_model.fit(X_train, y_train)

print("Decision Tree Classifier trained successfully!")

# Make predictions on the test set
# Similarly, use 'X_test' for prediction.
y_pred_dt = dt_model.predict(X_test)

# Evaluate the model
accuracy_dt = accuracy_score(y_test, y_pred_dt)
report_dt = classification_report(y_test, y_pred_dt)

print(f"\nDecision Tree Accuracy: {accuracy_dt:.4f}")
print("\nDecision Tree Classification Report:")
print(report_dt)

Decision Tree Classifier trained successfully!

Decision Tree Accuracy: 0.6763

Decision Tree Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.79      0.78       440
           1       0.57      0.56      0.57       293
           2       0.44      0.42      0.43        67

    accuracy                           0.68       800
   macro avg       0.60      0.59      0.59       800
weighted avg       0.67      0.68      0.67       800



#### 2. Support Vector Machine (SVM)

In [17]:
from sklearn.svm import SVC

# Initialize the SVM model
svm_model = SVC(kernel='linear', random_state=42) # Using a linear kernel for text data often works well

# Train the model
# The error occurred because 'X_train_tfidf' is not defined. We should use 'X_train' for this numerical dataset.
svm_model.fit(X_train, y_train)

print("SVM model trained successfully!")

# Make predictions on the test set
# Similarly, use 'X_test' for prediction.
y_pred_svm = svm_model.predict(X_test)

# Evaluate the model
accuracy_svm = accuracy_score(y_test, y_pred_svm)
report_svm = classification_report(y_test, y_pred_svm)

print(f"\nSVM Accuracy: {accuracy_svm:.4f}")
print("\nSVM Classification Report:")
print(report_svm)

SVM model trained successfully!

SVM Accuracy: 0.7963

SVM Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.89      0.87       440
           1       0.73      0.71      0.72       293
           2       0.72      0.58      0.64        67

    accuracy                           0.80       800
   macro avg       0.77      0.73      0.74       800
weighted avg       0.79      0.80      0.79       800



#### 3. K-Nearest Neighbors (KNN)

In [19]:
from sklearn.neighbors import KNeighborsClassifier

# Initialize the KNN model
# Adjusted n_neighbors to be <= n_samples_fit (3 in this case) due to small dataset
knn_model = KNeighborsClassifier(n_neighbors=3) # You can adjust n_neighbors

# Train the model
# The error occurred because 'X_train_tfidf' is not defined. We should use 'X_train' for this numerical dataset.
knn_model.fit(X_train, y_train)

print("KNN model trained successfully!")

# Make predictions on the test set
# Similarly, use 'X_test' for prediction.
y_pred_knn = knn_model.predict(X_test)

# Evaluate the model
accuracy_knn = accuracy_score(y_test, y_pred_knn)
report_knn = classification_report(y_test, y_pred_knn)

print(f"\nKNN Accuracy: {accuracy_knn:.4f}")
print("\nKNN Classification Report:")
print(report_knn)

KNN model trained successfully!

KNN Accuracy: 0.5425

KNN Classification Report:
              precision    recall  f1-score   support

           0       0.60      0.75      0.67       440
           1       0.43      0.33      0.38       293
           2       0.20      0.07      0.11        67

    accuracy                           0.54       800
   macro avg       0.41      0.39      0.39       800
weighted avg       0.51      0.54      0.52       800



### Loading a Movie Dataset

Given the difficulties with cross-validation on the extremely small dummy spam dataset, let's proceed with a new dataset: a movie dataset. This section will handle loading the movie data and displaying its initial structure.

In [24]:
import pandas as pd

# Attempt to load a movie dataset from Google Drive
# You might need to adjust the path if your movie dataset has a different name or location.
try:
    # The path was missing quotes, causing a SyntaxError. It should be a string.
    # The previous ParserError indicated a malformed CSV, likely an unclosed quote or line break within a field.
    # Using engine='python' and on_bad_lines='skip' to handle parsing errors.
    movie_df = pd.read_csv("/content/movies_metadata.csv", engine='python', on_bad_lines='skip') # Common name, adjust if needed
    print("Movie dataset loaded successfully!")
except FileNotFoundError:
    print("Error: movies.csv not found in MyDrive. Please check the path or filename.")
    print("If your movie dataset has a different name or is in a different folder, please update the path above.")
    # As a placeholder, creating a small dummy movie DataFrame if file not found
    data_placeholder = {
        'title': ['Movie A', 'Movie B', 'Movie C', 'Movie D', 'Movie E'],
        'genre': ['Action', 'Comedy', 'Drama', 'Action', 'Sci-Fi'],
        'rating': [7.5, 6.2, 8.9, 7.8, 9.1],
        'year': [2000, 2010, 2005, 2015, 1999]
    }
    movie_df = pd.DataFrame(data_placeholder)
    print("Using a dummy movie DataFrame for demonstration.")

# Display the first few rows and column information of the movie dataset
print("\nMovie Dataset Head:")
display(movie_df.head())
print("\nMovie Dataset Info:")
movie_df.info()

Movie dataset loaded successfully!

Movie Dataset Head:


,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0



Movie Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44130 entries, 0 to 44129
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  44130 non-null  object 
 1   belongs_to_collection  4415 non-null   object 
 2   budget                 44130 non-null  object 
 3   genres                 44130 non-null  object 
 4   homepage               7615 non-null   object 
 5   id                     44130 non-null  object 
 6   imdb_id                44114 non-null  object 
 7   original_language      44122 non-null  object 
 8   original_title         44130 non-null  object 
 9   overview               43233 non-null  object 
 10  popularity             44125 non-null  object 
 11  poster_path            43775 non-null  object 
 12  production_companies   44127 non-null  object 
 13  production_countries   44127 non-null  object 
 14  release_date           44052 non-

### Preparing Movie Data for GridSearchCV

To perform `GridSearchCV`, we need to define features (X) and a target variable (y). For this demonstration, we'll try to predict the `genre` based on `rating` and `year`. Since `genre` is categorical, we'll encode it numerically.

In [25]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

# Handle missing values in 'genres' and 'release_date' before processing
movie_df.dropna(subset=['genres', 'release_date', 'vote_average'], inplace=True)

# Encode 'genres' column to numerical labels
le = LabelEncoder()
movie_df['genre_encoded'] = le.fit_transform(movie_df['genres'])

# Extract year from 'release_date'
movie_df['release_year'] = pd.to_datetime(movie_df['release_date']).dt.year

# Define features (X) and target (y)
X_movie = movie_df[['vote_average', 'release_year']]
y_movie = movie_df['genre_encoded']

# Split the movie dataset into training and testing sets
# We will try to stratify, but due to many unique genres, some classes may still have few samples.
# This dataset is much larger, so stratification should generally work better than with the dummy data.
# Temporarily setting stratify=None to bypass the error caused by single-sample classes.
# For a more robust solution, rare classes in y_movie would need to be handled (e.g., filtered or grouped).
X_train_movie, X_test_movie, y_train_movie, y_test_movie = train_test_split(
    X_movie, y_movie, test_size=0.2, random_state=42, stratify=None # Changed to None to resolve ValueError
)

print(f"Movie Training set size: {len(X_train_movie)}")
print(f"Movie Testing set size: {len(X_test_movie)}")
print(f"\ny_train_movie value counts (top 10):\n{y_train_movie.value_counts().head(10)}")
print(f"y_test_movie value counts (top 10):\n{y_test_movie.value_counts().head(10)}")

Movie Training set size: 35239
Movie Testing set size: 8810

y_train_movie value counts (top 10):
genre_encoded
1972    3925
3145    2783
3991    2088
0       1829
1490    1015
2991     876
2146     767
2814     720
2927     482
1802     414
Name: count, dtype: int64
y_test_movie value counts (top 10):
genre_encoded
1972    945
3145    694
3991    514
0       437
1490    259
2991    230
2814    188
2146    182
2100    113
1802    111
Name: count, dtype: int64


### Running GridSearchCV on Movie Dataset (with anticipated limitations)

Now, let's attempt to run `GridSearchCV` on this prepared movie dataset. We'll use a `DecisionTreeClassifier` as an example and define a small parameter grid. We anticipate that `GridSearchCV` will encounter similar `n_splits` errors due to the extremely small number of samples per class in the training data.

In [26]:
# Define the model
dt_model_movie = DecisionTreeClassifier(random_state=42)

# Define a simple parameter grid
param_grid_movie = {
    'max_depth': [None, 2, 3],
    'min_samples_split': [2, 3]
}

# Initialize GridSearchCV
# We will use cv=2, but it's highly likely to fail because the training data (4 samples)
# has classes with only one instance (e.g., genre_encoded 0, 1, 2, 3 may each have 1 sample in y_train_movie).
# Sklearn's GridSearchCV with default StratifiedKFold for classification will struggle.
# For a real dataset, you would use a higher, meaningful cv value (e.g., 5 or 10).

try:
    grid_search_movie = GridSearchCV(dt_model_movie, param_grid_movie, cv=2, scoring='accuracy', n_jobs=-1)
    grid_search_movie.fit(X_train_movie, y_train_movie)

    print("GridSearchCV completed successfully!")
    print(f"Best parameters: {grid_search_movie.best_params_}")
    print(f"Best cross-validation score: {grid_search_movie.best_score_:.4f}")

    # Evaluate on the test set
    y_pred_grid_movie = grid_search_movie.predict(X_test_movie)
    print("\nClassification Report on Test Set:")
    print(classification_report(y_test_movie, y_pred_grid_movie))

except Exception as e:
    print(f"An error occurred during GridSearchCV: {e}")
    print("This is expected for such a small and imbalanced dummy dataset.")
    print("Specifically, with only 4 training samples and 4 unique genres,")
    print("it's impossible to create 2 folds where each class is sufficiently represented.")
    print("For a proper GridSearchCV, a larger and more balanced dataset is required.")

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=2.
  warnings.warn(


GridSearchCV completed successfully!
Best parameters: {'max_depth': 3, 'min_samples_split': 2}
Best cross-validation score: 0.1245

Classification Report on Test Set:
              precision    recall  f1-score   support

           0       0.24      0.19      0.21       437
           2       0.00      0.00      0.00         1
           5       0.00      0.00      0.00         1
           6       0.00      0.00      0.00         1
           7       0.00      0.00      0.00         1
           9       0.00      0.00      0.00         7
          11       0.00      0.00      0.00         1
          18       0.00      0.00      0.00         1
          21       0.00      0.00      0.00         1
          25       0.00      0.00      0.00         1
          30       0.00      0.00      0.00         1
          32       0.00      0.00      0.00         1
          34       0.00      0.00      0.00         1
          35       0.00      0.00      0.00         1
          36       0.0

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
